# GoMeal ML Brain And Feed Flow

This notebook is the high-level map for the GoMeal ML ranking system shown in your project tree.

It connects these modules:

- `brain/api/subscriber.py`: receives user action events and updates the brain
- `brain/core/network.py`: stores neurons, macro neurons, synapses, and patterns
- `brain/core/core/neuron.py`: user and recipe/post neurons
- `brain/core/core/macro_neuron.py`: abstractions over repeated patterns
- `brain/core/patterns/pattern_manager.py`: repeated activation tracking
- `routes/feed/rank/rank.py`: personalized feed ranking
- `routes/feed/trending/trend.py`: trending feed ranking
- `routes/feed/rank/diversify.py`: final diversity pass
- `routes/feed/scopes/scope.py`: scope tags for filtering and clustering

The core idea:

```text
User actions create graph memory.
Graph memory improves rank and trend feeds.
Rank and trend feeds still use embeddings, seen penalties, and diversity.
```

## Runtime Flow

```text
Node.js publishes user action
        |
        v
brain/api/subscriber.py
        |
        |-- get/create UserNeuron
        |-- get/create RecipeNeuron
        |-- connect user -> recipe
        |-- register user+recipe personal pattern
        |-- register recipe+recipe collaborative pattern
        |-- update user_vec in Redis/Postgres
        v
brain/core/network.py
        |
        |-- neurons
        |-- macro_neurons
        |-- synapses
        |-- pattern_manager
        |-- user_recent_recipe_activations
        |-- recipe_coactivation_counts
```

Then feed ranking reads the learned state:

```text
routes/feed/rank/rank.py
        |
        |-- semantic score: user_vec dot post_vec
        |-- direct brain boost: user -> recipe
        |-- collaborative boost: viewer recent recipe -> candidate recipe
        |-- seen penalty
        |-- diversity pass
```

Trending uses a similar final layer:

```text
routes/feed/trending/trend.py
        |
        |-- weighted global action score
        |-- recency decay
        |-- light user_vec personalization
        |-- collaborative boost
        |-- seen penalty
        |-- diversity pass
```

## Main Signals

| Signal | Where It Is Learned | Where It Is Used | Meaning |
|---|---|---|---|
| `user_vec` | `subscriber.py` | `rank.py`, `trend.py` | personal semantic taste |
| direct synapse | `subscriber.py` / `network.py` | `rank.py` | this user interacted with this recipe |
| user+recipe macro | `network.py` | later memory analysis | repeated personal activation |
| recipe+recipe coactivation | `network.py` | `rank.py`, `trend.py` | multi-user collaborative signal |
| scope tags | `scope.py`, embed pipeline | candidate filtering/clustering | food category and intent |
| diversity penalty | `diversify.py` | final feed pass | prevents near-duplicate feed results |

## Recommended Notebook Order

1. `gomeal_ml_00_system_flow.ipynb`: this map.
2. `gomeal_ml_01_brain_learning.ipynb`: how neurons, patterns, macro neurons, and multi-user coactivation work.
3. `gomeal_ml_02_rank_and_trend_integration.ipynb`: how to update `_rank` and `_post_trends`.
4. `gomeal_ml_03_scopes_and_diversity.ipynb`: how scope tags and diversity shape final feed quality.